step 1
create full wide csv files for train and test

In [44]:
from pathlib import Path
import csv
import pandas as pd

DATA_DIR = Path("EURLex-4K")
TRN_FT = DATA_DIR / "trn_ft_mat.txt"
TRN_LBL = DATA_DIR / "trn_lbl_mat.txt"
TST_FT = DATA_DIR / "tst_ft_mat.txt"
OUT_TRAIN_WIDE = Path("eurlex.csv")
OUT_TEST_WIDE = Path("eurlex_test.csv")

for p in [TRN_FT, TRN_LBL, TST_FT]:
    if not p.exists():
        raise FileNotFoundError(f"missing file: {p}")

print("files ready")

files ready


In [45]:
with TRN_FT.open("r", encoding="utf-8") as f:
    trn_ft_header = f.readline().strip()
    trn_ft_rows = f.readlines()

with TRN_LBL.open("r", encoding="utf-8") as f:
    trn_lbl_header = f.readline().strip()
    trn_lbl_rows = f.readlines()

trn_n, trn_feat_cols = map(int, trn_ft_header.split())
trn_lbl_n, trn_lbl_cols = map(int, trn_lbl_header.split())

if trn_n != trn_lbl_n:
    raise ValueError("train row mismatch")

print(f"train rows: {trn_n}")
print(f"feature cols: {trn_feat_cols}")
print(f"label cols: {trn_lbl_cols}")

train rows: 15377
feature cols: 5000
label cols: 3993


In [46]:
feature_cols = [f"f{j}" for j in range(trn_feat_cols)]

with OUT_TRAIN_WIDE.open("w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["row_id", *feature_cols, "labels"])

    for i in range(trn_n):
        dense = [0.0] * trn_feat_cols
        feat_text = trn_ft_rows[i].strip()
        lbl_text = trn_lbl_rows[i].strip()

        if feat_text:
            for token in feat_text.split():
                idx_str, val_str = token.split(":", 1)
                dense[int(idx_str)] = float(val_str)

        writer.writerow([i, *dense, lbl_text])

print(f"saved: {OUT_TRAIN_WIDE}")
print(f"train rows written: {trn_n}")

saved: eurlex.csv
train rows written: 15377


In [47]:
with OUT_TEST_WIDE.open("w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["row_id", *feature_cols])

    for i in range(tst_n):
        dense = [0.0] * trn_feat_cols
        feat_text = tst_ft_rows[i].strip()

        if feat_text:
            for token in feat_text.split():
                idx_str, val_str = token.split(":", 1)
                dense[int(idx_str)] = float(val_str)

        writer.writerow([i, *dense])

print(f"saved: {OUT_TEST_WIDE}")
print(f"test rows written: {tst_n}")

saved: eurlex_test.csv
test rows written: 3971


In [48]:
train_loaded = pd.read_csv(OUT_TRAIN_WIDE, nrows=2)
test_loaded = pd.read_csv(OUT_TEST_WIDE, nrows=2)

print(f"train columns: {len(train_loaded.columns)}")
print(f"test columns: {len(test_loaded.columns)}")
print(train_loaded.iloc[:2, :8])
print(test_loaded.iloc[:2, :8])

train columns: 5002
test columns: 5001
   row_id   f0   f1   f2   f3   f4   f5   f6
0       0  0.0  0.0  0.0  0.0  0.0  0.0  0.0
1       1  0.0  0.0  0.0  0.0  0.0  0.0  0.0
   row_id       f0   f1   f2   f3   f4   f5   f6
0       0  0.00000  0.0  0.0  0.0  0.0  0.0  0.0
1       1  3.62294  0.0  0.0  0.0  0.0  0.0  0.0
